In [2]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "src").exists():
    PROJECT_DIR = PROJECT_DIR.parent

sys.path.insert(0, str(PROJECT_DIR))

from src.image_classification.data_loader import load_datasets

DATA_DIR = PROJECT_DIR / "data" / "Vegetables" / "Vegetable_Images"

train_ds, val_ds, test_ds = load_datasets(DATA_DIR)

Found 4000 files belonging to 4 classes.
Found 800 files belonging to 4 classes.
Found 800 files belonging to 4 classes.


In [3]:
import sys
import tensorflow as tf

print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)

Python: c:\Users\fondo\AppData\Local\Programs\Python\Python311\python.exe
TensorFlow: 2.21.0


In [5]:

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    GlobalAveragePooling2D,
    Dense,
    Dropout
)

# Define the CNN architecture
model = Sequential([
    Input(shape=(224, 224, 3)),

    # First convolutional block
    Conv2D(32, kernel_size=3, activation="relu"),
    MaxPooling2D(pool_size=2),

    # Second convolutional block
    Conv2D(64, kernel_size=3, activation="relu"),
    MaxPooling2D(pool_size=2),

    # Third convolutional block
    Conv2D(128, kernel_size=3, activation="relu"),
    MaxPooling2D(pool_size=2),

    # Classification head
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.3),

    # Four output classes
    Dense(4, activation="softmax")
])

# Display the architecture
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,276 (430.77 KB)

 Trainable params: 110,276 (430.77 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("CNN compiled successfully!")


CNN compiled successfully!


In [10]:
from pathlib import Path

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Model directory ready:", MODEL_DIR.resolve())

Model directory ready: C:\Users\fondo\deep_learning\dl_tensorflow\AI_Prediction_Platform\notebooks\models


In [11]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

callbacks = [
    ModelCheckpoint(
        filepath=str(MODEL_DIR / "best_vegetable_cnn.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

In [7]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/12


c:\Users\fondo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step - accuracy: 0.4692 - loss: 1.1458
Epoch 1: val_loss improved from None to 0.94472, saving model to models\best_vegetable_cnn.keras

Epoch 1: finished saving model to models\best_vegetable_cnn.keras
500/500 ━━━━━━━━━━━━━━━━━━━━ 108s 212ms/step - accuracy: 0.4692 - loss: 1.1458 - val_accuracy: 0.6275 - val_loss: 0.9447 - learning_rate: 0.0010
Epoch 2/12
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 260ms/step - accuracy: 0.6783 - loss: 0.7730
Epoch 2: val_loss improved from 0.94472 to 0.57379, saving model to models\best_vegetable_cnn.keras

Epoch 2: finished saving model to models\best_vegetable_cnn.keras
500/500 ━━━━━━━━━━━━━━━━━━━━ 140s 280ms/step - accuracy: 0.6783 - loss: 0.7730 - val_accuracy: 0.7788 - val_loss: 0.5738 - learning_rate: 0.0010
Epoch 3/12
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 267ms/step - accuracy: 0.8150 - loss: 0.5009
Epoch 3: val_loss improved from 0.57379 to 0.31965, saving model to models\best_vegetable_cnn.keras

Epoch 3: finished saving 

In [16]:
from tensorflow.keras.models import load_model

MODEL_PATH = PROJECT_DIR / "models" / "best_vegetable_cnn.keras"

if not MODEL_PATH.exists():
    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    model.save(MODEL_PATH)

best_model = load_model(MODEL_PATH)

print("Model loaded successfully.")

Model loaded successfully.


In [18]:
best_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

test_loss, test_accuracy = best_model.evaluate(test_ds)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4%}")

100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 67ms/step - accuracy: 0.2500 - loss: 1.3867
Test Loss: 1.3867
Test Accuracy: 25.0000%


In [19]:
# Check class mappings
print("Class mapping:", train_ds.class_names if hasattr(train_ds, "class_names") else "Not exposed")
print("Model output shape:", best_model.output_shape)

# Evaluate all three datasets using the same saved model
print("\n--- TRAIN ---")
best_model.evaluate(train_ds, verbose=1)

print("\n--- VALIDATION ---")
best_model.evaluate(val_ds, verbose=1)

print("\n--- TEST ---")
best_model.evaluate(test_ds, verbose=1)

Class mapping: Not exposed
Model output shape: (None, 4)

--- TRAIN ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 35s 69ms/step - accuracy: 0.2500 - loss: 1.3869

--- VALIDATION ---
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 67ms/step - accuracy: 0.2500 - loss: 1.3865

--- TEST ---
100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 64ms/step - accuracy: 0.2500 - loss: 1.3867


[1.386733055114746, 0.25]

In [20]:
import numpy as np

# Take one batch of test images
images, labels = next(iter(test_ds))

# Compare predictions from both models
original_predictions = model.predict(images[:5], verbose=0)
loaded_predictions = best_model.predict(images[:5], verbose=0)

print("Original model predictions:")
print(np.round(original_predictions, 3))

print("\nLoaded model predictions:")
print(np.round(loaded_predictions, 3))

print("\nOriginal model confidence:")
print(original_predictions.max(axis=1))

print("\nLoaded model confidence:")
print(loaded_predictions.max(axis=1))

Original model predictions:
[[0.247 0.252 0.255 0.246]
 [0.245 0.251 0.257 0.247]
 [0.245 0.251 0.257 0.248]
 [0.246 0.251 0.255 0.248]
 [0.246 0.252 0.256 0.247]]

Loaded model predictions:
[[0.247 0.252 0.255 0.246]
 [0.245 0.251 0.257 0.247]
 [0.245 0.251 0.257 0.248]
 [0.246 0.251 0.255 0.248]
 [0.246 0.252 0.256 0.247]]

Original model confidence:
[0.25548327 0.2569345  0.25658706 0.2549723  0.25578645]

Loaded model confidence:
[0.25548327 0.2569345  0.25658706 0.2549723  0.25578645]
